In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

In [ ]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

In [ ]:
df.show(10, truncate=False)

In [ ]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

In [ ]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

In [ ]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

In [ ]:
hourly.printSchema()

In [ ]:
from pyspark.sql.functions import window

min30 = (
    df.groupBy(window("timestamp", "30 minutes"), "store")    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window","store")
)
min30.show(truncate=False)

In [ ]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

In [ ]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ: ponieważ okna w sliding moga na siebie "nachodzic" (jedna transakcja moze byc w dwoch oknach)


In [ ]:
from pyspark.sql.functions import col, avg, round as _round, hour

gda_hourly_avg = (
    df.filter(col("store") == "Gdańsk")
    .withColumn("hour", hour(col("timestamp")))
    .groupBy("hour")
    .agg(
        _round(avg("amount"), 2).alias("srednia_PLN")
    )
    .orderBy("srednia_PLN")
)

gda_hourly_avg.show()

In [ ]:
gda_hourly_avg.head(1)

In [ ]:
from pyspark.sql.functions import col, count

result = (
    df.filter(
        (col("timestamp").substr(12, 5) >= "09:00") &
        (col("timestamp").substr(12, 5) < "09:30")
    )
    .groupBy("category")
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .orderBy("category")
)

result.show()

In [ ]:
from pyspark.sql.functions import window, count, col

quarter_peak = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .orderBy(col("liczba_tx").desc())
)

quarter_peak.show(truncate=False)